<a href="https://colab.research.google.com/github/sametz/BCCE2026/blob/main/molviz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install rdkit py3Dmol

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 36.7 MB/s eta 0:00:00


In [5]:
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
import py3Dmol
import numpy as np

# Define the SMILES string for methylcyclohexane
smiles = 'CC1CCCCC1'
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # Add hydrogens to get proper 3D geometry

# Generate multiple conformers
num_confs = 50 # Generate a good number of conformers to find both types
# Use a consistent random seed for reproducibility of the initial conformer set
AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, randomSeed=42, numThreads=0)
# Optimize all generated conformers using UFF
AllChem.UFFOptimizeMoleculeConfs(mol)

equatorial_conf = None
axial_conf = None

# Identify the methyl carbon (atom index 0) and the ring carbon it's attached to (atom index 1)
methyl_carbon_idx = 0
ring_attachment_idx = 1

# Helper function to classify the methyl group as axial or equatorial
def is_axial_methyl(mol_obj, conformer):
    # Get positions of methyl carbon and its attachment point
    methyl_pos = conformer.GetAtomPosition(methyl_carbon_idx)
    attachment_pos = conformer.GetAtomPosition(ring_attachment_idx)

    # Get the neighbors of the attachment point (excluding the methyl carbon)
    attachment_atom = mol_obj.GetAtomWithIdx(ring_attachment_idx)
    ring_neighbors_of_attachment = [
        a.GetIdx() for a in attachment_atom.GetNeighbors() if a.GetIdx() != methyl_carbon_idx
    ]

    if len(ring_neighbors_of_attachment) < 2:
        # Should have at least two ring neighbors for a cyclohexane
        return False, None

    # Get positions of two ring neighbors of the attachment carbon
    neighbor_pos1 = conformer.GetAtomPosition(ring_neighbors_of_attachment[0])
    neighbor_pos2 = conformer.GetAtomPosition(ring_neighbors_of_attachment[1])

    # Calculate vectors for the C-C bonds of the attachment carbon within the ring
    vec1 = np.array([neighbor_pos1.x - attachment_pos.x, neighbor_pos1.y - attachment_pos.y, neighbor_pos1.z - attachment_pos.z])
    vec2 = np.array([neighbor_pos2.x - attachment_pos.x, neighbor_pos2.y - attachment_pos.y, neighbor_pos2.z - attachment_pos.z])

    # Calculate the cross product to get a vector approximately normal to the local ring plane at the attachment point.
    # This vector approximates the axial direction of the C-H bonds at this carbon.
    local_normal_vector = np.cross(vec1, vec2)
    local_normal_vector = local_normal_vector / np.linalg.norm(local_normal_vector) # Normalize the normal vector

    # Vector from attachment point to methyl carbon
    vec_methyl = np.array([methyl_pos.x - attachment_pos.x, methyl_pos.y - attachment_pos.y, methyl_pos.z - attachment_pos.z])
    vec_methyl = vec_methyl / np.linalg.norm(vec_methyl) # Normalize the methyl vector

    # Calculate the dot product to see how parallel vec_methyl is to the local_normal_vector
    # A dot product close to 1 or -1 means parallel (axial).
    # A dot product close to 0 means perpendicular (equatorial).
    dot_product = np.dot(vec_methyl, local_normal_vector)

    # Heuristic thresholds for classification
    axial_threshold = 0.7       # If |dot_product| > 0.7, consider axial
    equatorial_threshold = 0.4  # Relaxed: If |dot_product| < 0.4, consider equatorial

    if abs(dot_product) > axial_threshold:
        return True, dot_product # It's axial
    elif abs(dot_product) < equatorial_threshold:
        return False, dot_product # It's equatorial
    else:
        return None, dot_product # Ambiguous, or neither clearly axial nor equatorial

# Iterate through conformers to find one axial and one equatorial
for i in range(mol.GetNumConformers()):
    conf = mol.GetConformer(i)
    is_axial, dp = is_axial_methyl(mol, conf)

    if is_axial is True and axial_conf is None:
        axial_conf = conf
    elif is_axial is False and equatorial_conf is None:
        equatorial_conf = conf

    if axial_conf is not None and equatorial_conf is not None:
        break # Found both, exit loop

# Fallback if specific conformers weren't clearly identified by the heuristic
if equatorial_conf is None:
    print("Warning: Could not find a clear equatorial conformer using the heuristic. Using the first conformer.")
    equatorial_conf = mol.GetConformer(0)

if axial_conf is None:
    print("Warning: Could not find a clear axial conformer using the heuristic. Attempting to use a distinct conformer if available.")
    # Try to find any conformer different from the selected equatorial one
    found_distinct_for_axial = False
    for i in range(mol.GetNumConformers()):
        if mol.GetConformer(i).GetId() != equatorial_conf.GetId():
            axial_conf = mol.GetConformer(i)
            found_distinct_for_axial = True
            print("Using a distinct conformer for axial display, but it might not be truly axial.")
            break
    if not found_distinct_for_axial:
        axial_conf = mol.GetConformer(0) # Fallback to first if no other distinct conformer was found

# Create new RDKit molecule objects for display, each with only its selected conformer
mol_equatorial_display = Chem.Mol(mol)
mol_equatorial_display.RemoveAllConformers()
mol_equatorial_display.AddConformer(equatorial_conf, assignId=True)

mol_axial_display = Chem.Mol(mol)
mol_axial_display.RemoveAllConformers()
mol_axial_display.AddConformer(axial_conf, assignId=True)

# Create a 3Dmol viewer with a 1x2 grid for side-by-side display
view = py3Dmol.view(query='_ALL_', linked=False, viewergrid=(1, 2))

# Add the equatorial molecule to the first panel
# Convert RDKit molecule to MolBlock string for py3Dmol
view.addModel(Chem.MolToMolBlock(mol_equatorial_display), 'mol', viewer=(0,0))
view.setStyle({'stick':{}}, viewer=(0,0))
view.zoomTo(viewer=(0,0))
view.addLabel('Equatorial Conformation', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'}, {'viewer': [0, 0]})

# Add the axial molecule to the second panel
view.addModel(Chem.MolToMolBlock(mol_axial_display), 'mol', viewer=(0,1))
view.setStyle({'stick':{}}, viewer=(0,1))
view.zoomTo(viewer=(0,1))
view.addLabel('Axial Conformation', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'}, {'viewer': [0, 1]})

# Render the viewer
view.render()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.